# Voice-to-Task Agent — zero-install notebook

This notebook is a runnable, zero-install companion to the course project [Build a Voice-to-Task Agent](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/voice-to-task-agent) and the fuller example in [`examples/voice-to-task-agent/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/voice-to-task-agent).

It transcribes a short voice memo to text, entirely locally and for free, using OpenAI's *open-source* [Whisper](https://github.com/openai/whisper) model — **not** the paid Whisper API — then asks a free-tier LLM to turn the transcript into a structured task list (task / due date / priority).

Works unmodified on Colab, Kaggle, and Binder.

## Optional: use a GPU for faster transcription

Whisper's transcription speed scales a lot with hardware. If you're running this on **Google Colab**, you get a free GPU that a CPU-only laptop doesn't — go to **Runtime > Change runtime type > T4 GPU** before running the cells below. This is optional: the pipeline works fine on CPU too, just slower for bigger Whisper model sizes. (Kaggle also offers free GPU quota under **Settings > Accelerator** if you'd like to try that route there.)

In [ ]:
%pip install -q openai-whisper openai python-dotenv

## Get a sample voice memo

The course repo ships three short (~15-20s) synthetic voice-memo clips at [`examples/voice-to-task-agent/sample_audio/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/voice-to-task-agent/sample_audio) so this runs with zero setup — no microphone, no recording, no local files needed. The cell below downloads one directly from GitHub so it works the same way on Colab, Kaggle, and Binder.

Change `SAMPLE_AUDIO_FILENAME` to try one of the other two clips:
- `memo_1_work_followups.wav` — meeting follow-ups with a deadline and a "no rush" item
- `memo_2_personal_errands.wav` — personal errands, one flagged urgent
- `memo_3_project_planning.mp3` — project tasks mixing a high-priority blocker and a low-priority nice-to-have

In [ ]:
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/add-voice-to-task-agent-project/examples/voice-to-task-agent/sample_audio"
SAMPLE_AUDIO_FILENAME = "memo_1_work_followups.wav"  # try memo_2_personal_errands.wav or memo_3_project_planning.mp3

audio_path = SAMPLE_AUDIO_FILENAME
urllib.request.urlretrieve(f"{RAW_BASE}/{SAMPLE_AUDIO_FILENAME}", audio_path)
print(f"Downloaded {audio_path}")

Prefer to use your own recording instead? Uncomment the cell below to upload a file from your computer (Colab only — it's a no-op, skipped safely, on Kaggle/Binder).

In [ ]:
# Uncomment to upload your own audio file instead of the sample above (Colab only).
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     audio_path = next(iter(uploaded))
#     print(f"Using uploaded file: {audio_path}")
# except ImportError:
#     print("Not running on Colab -- keeping the downloaded sample audio file instead.")

## Step 1: Transcribe the voice memo locally

`whisper.load_model("base")` loads a neural network trained on a huge amount of multilingual speech data; `model.transcribe(audio_path)` runs it on the audio file and returns a dict whose `"text"` key is the full transcript. Whisper decodes the audio itself (via `ffmpeg` under the hood), so this works on both `.wav` and `.mp3` without any manual conversion.

The first call downloads the model weights (~140MB for `"base"`) and caches them, so only the first run is slow.

In [ ]:
import whisper

WHISPER_MODEL_SIZE = "base"  # tiny / base / small / medium / large -- bigger is more accurate and slower

_whisper_model = None


def get_whisper_model():
    global _whisper_model
    if _whisper_model is None:
        print(f"Loading Whisper '{WHISPER_MODEL_SIZE}' model (first run downloads it)...")
        _whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)
    return _whisper_model


def transcribe(audio_path: str) -> str:
    """Transcribes an audio file to plain text, entirely locally."""
    model = get_whisper_model()
    result = model.transcribe(audio_path)
    return result["text"].strip()


transcript = transcribe(audio_path)
print("\n--- Transcript ---")
print(transcript)

## Step 2: Set a free-tier LLM API key

Transcription above is local and free, no key needed. Extracting structured action items from the transcript needs a real hosted LLM — free tier, but you still need an API key from one provider. Pick any one of the six below (all expose an OpenAI-compatible endpoint, so the same client code works for all of them):

| Provider | Where to get a key | Env var |
|---|---|---|
| GitHub Models *(default)* | [github.com/settings/tokens](https://github.com/settings/tokens) (`models: read` scope) | `GITHUB_TOKEN` |
| Gemini | [aistudio.google.com](https://aistudio.google.com/) | `GOOGLE_API_KEY` |
| Groq | [console.groq.com/keys](https://console.groq.com/keys) | `GROQ_API_KEY` |
| Mistral | [console.mistral.ai/api-keys](https://console.mistral.ai/api-keys) | `MISTRAL_API_KEY` |
| Cerebras | [cloud.cerebras.ai](https://cloud.cerebras.ai/) | `CEREBRAS_API_KEY` |
| OpenRouter | [openrouter.ai/keys](https://openrouter.ai/keys) | `OPENROUTER_API_KEY` |

`getpass` below keeps the key out of the notebook's saved output and command history — set `LLM_PROVIDER` to match whichever provider's key you paste in.

In [ ]:
import os
from getpass import getpass

LLM_PROVIDER = "github"  # one of: github, gemini, groq, mistral, cerebras, openrouter

PROVIDERS = {
    "github": {"env": "GITHUB_TOKEN", "base_url": "https://models.github.ai/inference", "model": "gpt-4o-mini"},
    "gemini": {"env": "GOOGLE_API_KEY", "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", "model": "gemini-3.5-flash"},
    "groq": {"env": "GROQ_API_KEY", "base_url": "https://api.groq.com/openai/v1", "model": "llama-3.3-70b-versatile"},
    "mistral": {"env": "MISTRAL_API_KEY", "base_url": "https://api.mistral.ai/v1", "model": "mistral-small-latest"},
    "cerebras": {"env": "CEREBRAS_API_KEY", "base_url": "https://api.cerebras.ai/v1", "model": "llama-3.3-70b"},
    "openrouter": {"env": "OPENROUTER_API_KEY", "base_url": "https://openrouter.ai/api/v1", "model": "meta-llama/llama-3.3-70b-instruct:free"},
}

_env_var = PROVIDERS[LLM_PROVIDER]["env"]
os.environ[_env_var] = getpass(f"Paste your {_env_var} (input hidden): ")

## Step 3: Extract structured action items with the LLM

The prompt tells the model exactly what shape to return — a JSON object with a `"tasks"` list — and gives explicit rules for the tricky parts: don't invent a due date that was never said, don't guess a priority that isn't actually implied. `_parse_tasks_response` strips a stray markdown code fence if the model adds one despite being told not to, which free-tier models occasionally do.

In [ ]:
import json
from openai import OpenAI

EXTRACTION_PROMPT = """You extract action items from a voice memo transcript.

Read the transcript below and return a JSON object shaped exactly like this,
with no other text before or after it, and no markdown code fences:

{{"tasks": [{{"task": "...", "due_date": "...", "priority": "..."}}]}}

Rules:
- "task" is a short, clear description of what needs to be done, rewritten
  as an action (e.g. "Email the client the revised proposal"), not a raw
  quote from the transcript.
- "due_date" is null if the transcript doesn't mention one, otherwise a
  short phrase as stated (e.g. "Friday", "next week") -- do not invent a
  specific calendar date that was never said.
- "priority" is "high", "medium", or "low" if the transcript implies one
  (e.g. "urgent", "no rush"), otherwise null. Don't guess a priority that
  isn't actually implied.
- If the transcript contains no action items at all, return {{"tasks": []}}.

Transcript:
\"\"\"
{transcript}
\"\"\"
"""


def build_client(provider: str):
    config = PROVIDERS[provider]
    api_key = os.environ[config["env"]]
    client = OpenAI(api_key=api_key, base_url=config["base_url"])
    return client, config["model"]


def _parse_tasks_response(raw_text: str) -> list:
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text[4:] if text.startswith("json") else text
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as error:
        raise ValueError(f"Model reply wasn't valid JSON:\n{raw_text}") from error
    tasks = parsed.get("tasks")
    if tasks is None:
        raise ValueError(f"Model reply had no 'tasks' key:\n{raw_text}")
    return tasks


def extract_action_items(transcript: str, provider: str = LLM_PROVIDER) -> list:
    if not transcript.strip():
        return []
    client, model = build_client(provider)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(transcript=transcript)}],
    )
    return _parse_tasks_response(response.choices[0].message.content)

## Step 4: Run the whole pipeline end to end

Transcribe, extract, print a readable list with a priority marker per line, and save the result as `tasks.json` -- exactly what [`voice_to_tasks.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/voice-to-task-agent/voice_to_tasks.py) does when run as a script.

In [ ]:
def print_tasks(tasks: list) -> None:
    if not tasks:
        print("No action items found in this memo.")
        return
    markers = {"high": "\U0001f534", "medium": "\U0001f7e1", "low": "\U0001f7e2"}
    for item in tasks:
        marker = markers.get((item.get("priority") or "").lower(), "⚪")
        due = f" (due: {item['due_date']})" if item.get("due_date") else ""
        print(f"{marker} {item['task']}{due}")


print(f"Transcribing {audio_path} ...")
transcript = transcribe(audio_path)
print("\n--- Transcript ---")
print(transcript)

print("\nExtracting action items...")
tasks = extract_action_items(transcript)

print("\n--- Action items ---")
print_tasks(tasks)

with open("tasks.json", "w", encoding="utf-8") as f:
    json.dump(tasks, f, indent=2, ensure_ascii=False)
print(f"\nSaved {len(tasks)} task(s) to tasks.json")

## Where to go from here

- Change `SAMPLE_AUDIO_FILENAME` above and re-run to try the other two sample memos.
- Try a bigger `WHISPER_MODEL_SIZE` (`"small"` or `"medium"`) on a longer, messier recording -- this is exactly the kind of experiment the free Colab GPU is good for.
- See the full walkthrough in the [Build a Voice-to-Task Agent lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/voice-to-task-agent) for the reasoning behind each step, common pitfalls, and where to take this project next.